# Day 1 — MobileNetV2 compression
Enable a Kaggle GPU accelerator. Attach Kaggle Datasets containing `baseline.pt` and the CIFAR-10 data. The repository excludes checkpoints and raw data from Git.

Kaggle mounts the converted archive as `cifar-data/cifar-10-batches-py`. All commands below use that mounted directory directly, so torchvision does not attempt a live download.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("GITHUB_TOKEN")

username = "AdiGiriIIT"
repo_name = "CS6886--Assignment-2"
!git clone https://{token}@github.com/{username}/{repo_name}.git assignment-2

In [ ]:
from pathlib import Path

BASELINE_SOURCE = '/kaggle/input/datasets/adityagirishep23b048/baseline/baseline.pt'  # replace if needed
# This is the parent directory that contains cifar-10-batches-py.
DATA_DIR = '/kaggle/input/datasets/adityagirishep23b048/cifar-data'

%cd assignment-2
!python -m pip install -q PyYAML matplotlib
!mkdir -p results/checkpoints
!cp $BASELINE_SOURCE results/checkpoints/baseline.pt

cifar_dir = Path(DATA_DIR) / 'cifar-10-batches-py'
required_files = ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta']
missing_files = [name for name in required_files if not (cifar_dir / name).is_file()]
if missing_files:
    raise FileNotFoundError(f'Missing CIFAR-10 files under {cifar_dir}: {missing_files}')

print(f'Using Kaggle-local CIFAR-10 data: {cifar_dir}')
print('No CIFAR-10 download or extraction is needed.')
!sha256sum results/checkpoints/baseline.pt


In [ ]:
# Expected hash: 02ff38ac832c9fa3d72cad3db375b1103991b64eaf417c796ab352b49fbaae3d
!python -m unittest discover -s tests -v
!python -m src.evaluate --checkpoint results/checkpoints/baseline.pt --data-dir "$DATA_DIR" --device cuda


In [ ]:
# PTQ diagnostic and coverage audit using Kaggle-local CIFAR-10 data.
!python -m src.compress --checkpoint results/checkpoints/baseline.pt --data-dir "$DATA_DIR" --weight-bits 8 --activation-bits 8 --evaluate --device cuda
!python -m src.compress --checkpoint results/checkpoints/baseline.pt --data-dir "$DATA_DIR" --weight-bits 6 --activation-bits 6 --evaluate --device cuda


In [ ]:
# Short QAT pilots using Kaggle-local CIFAR-10 data.
!python -m src.qat --checkpoint results/checkpoints/baseline.pt --data-dir "$DATA_DIR" --weight-bits 8 --activation-bits 8 --epochs 2 --device cuda
!python -m src.qat --checkpoint results/checkpoints/baseline.pt --data-dir "$DATA_DIR" --weight-bits 6 --activation-bits 6 --epochs 2 --device cuda


In [ ]:
# Preserve notebook outputs for download.
!tar -czf day1_artifacts.tgz results/tables
from IPython.display import FileLink
FileLink('day1_artifacts.tgz')
